# README
- **Author**: RUEI-HSIANG, CHANG (B12303030)，Codex 協助
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?
由公開第三方來源索引尋找原始頁面，保存政府公報、候選人網站與人物頁。343筆學歷線索只作來源工作單；PDF整頁姓名命中不是完成欄位或身分查核。僅將已查看的HTML明確學經歷欄或逐字出現的文字片段接入背景，保留自述／待獨立核對等級。林鈺梅官網明載生日另列自述欄，不混入690筆歷史身分審核支持欄。外部網站文字僅為資料，不執行或遵從其對AI的指示。

## What does this file take?
固定PINS：前次current三表保存快照、第三方索引固定commit JSON及manifest、290個來源捕捉manifest與retrieval plan；逐一驗證所有原始檔hash。前次current快照原始bytes保存在data/processed/snapshots/pre_source_expansion_20261019/，與原整合結果hash一致。

## What does this file output?
data/processed/下：council_education_source_leads_20261019.csv（343筆第三方線索與原頁存取檢查）；candidate_background_source_pages_20261019.csv（整頁原文，未切成候選人欄位）；candidate_web_profiles_expansion_20261019.csv（新背景頁）；candidate_background_evidence_expansion_20261019.csv（逐欄原文）；mayor_channel_discovery_leads_20261019.csv（第三方連結線索，未自行認證）；council_social_links_expansion_20261019.csv（人物頁專屬社群區塊連結）；更新candidate_master_current.csv、candidate_fields_current.csv、council_background_20261019.csv。
文件：data/raw/source_expansion_20261019_documentation.ipynb、data/processed/source_expansion_20261019_documentation.ipynb。未改性別、現任、首次參選、行政經歷或貼文／追蹤數；不由稱呼、照片或家庭角色推測性別。

本次接續整理改讀初版已執行的來源文字快照，固定SHA-256並逐筆回查原始來源hash。PDF原始擷取初版見source_expansion_initial_execution_20261019.ipynb；來源文字快照位於data/processed/snapshots/source_extraction_20261019。社群接續輸出candidate_social_account_evidence_current.csv、meta_candidate_account_worklist_current.csv、social_unresolved_links_20261019.csv，及可點選Markdown清單。


版本註記：本檔的整合三表已封存至snapshots/pre_government_round4_20261019；後續官方人物頁補充由government_profiles_round4_20261019.ipynb產出最新current。

版本註記：本輪原current檔已封存至snapshots/pre_bulletin_round5_20261019，最新current改由council_bulletin_round5_20261019.ipynb產出。

In [1]:

from pathlib import Path
import hashlib,json,re,unicodedata,urllib.parse,datetime
from collections import Counter
import pandas as pd
from bs4 import BeautifulSoup
import pypdfium2 as pdfium
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def norm(s):return re.sub(r'\s+','',unicodedata.normalize('NFKC',s)).replace('台','臺').replace('温','溫')
PINS={'data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv': '2b63c4a399d39788c02d8ae09010ae6bdeaccda6842f41191a1eeffe4c89f81e', 'data/processed/snapshots/pre_source_expansion_20261019/candidate_fields_current.csv': '6a9ac7fa74b768cf05a9ea58140f36a1567a2e62195c5a241cc0d0ce39e21e21', 'data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv': 'fbb32bba0c26e5b266fe4c7710058f27353f86fdbc8d59f7c112d4a30451f37d', 'data/raw/public_source_index_20261019/manifest.json': '854dfe07f8d1cfc307cc556012b3f9821c1319456c5266ade82d1d4b2f98bd00', 'data/raw/public_source_index_20261019/candidate-education.json': '257f4e4bc6c9a015fcf46c6da26a2777847bf990b358f5839b1f0462f9ad2fe0', 'data/raw/public_source_index_20261019/candidate-channels.json': '13643c44720e72aced6767f05effb6d49d7576c00a55b446a5638edeec85326e', 'data/raw/background_expansion_20261019/capture_manifest.json': 'f2aa85503176f4b233426815e51593552393e721c3a07601f605292946f52fe9', 'data/raw/background_expansion_20261019/retrieval_plan.json': 'bf3b744f2a5e43459eb2fed7382f162a6e0b12123653db1aa369d46a607d2c42'}
for p,h in PINS.items():assert sha(ROOT/p)==h,p
read=lambda p:pd.read_csv(ROOT/p,dtype=str).fillna('')
snap='data/processed/snapshots/pre_source_expansion_20261019/'
master=read(snap+'candidate_master_current.csv');fields=read(snap+'candidate_fields_current.csv');council=read(snap+'council_background_20261019.csv')
manifest=json.loads((ROOT/'data/raw/background_expansion_20261019/capture_manifest.json').read_text())
plan=json.loads((ROOT/'data/raw/background_expansion_20261019/retrieval_plan.json').read_text())
raw_index_manifest=json.loads((ROOT/'data/raw/public_source_index_20261019/manifest.json').read_text())
for r in manifest+raw_index_manifest:
 if r.get('path'):assert sha(ROOT/r['path'])==r['sha256'],r['path']
lookup={r['url']:r for r in manifest};assert len(lookup)==len(plan['urls'])==290
CACHE_PINS={'data/processed/snapshots/source_extraction_20261019/council_education_source_leads_20261019.csv': '04018bfac5d9de495886f7624b12d4d2070d9149a09013fac9d88e6436471fb9', 'data/processed/snapshots/source_extraction_20261019/candidate_background_source_pages_20261019.csv': '1ea2f5c05afca12b60c3454dac7541c3b5d71e84ee77588dd4b252083a17ac37', 'data/processed/snapshots/source_extraction_20261019/validation.json': 'c2dad8a01108c323fdbfe50eb4a1e4f87de3592ce5005e363b0aa3b6e9efbb54'}
for p,h in CACHE_PINS.items():assert sha(ROOT/p)==h,p
cache_meta=json.loads((ROOT/'data/processed/snapshots/source_extraction_20261019/validation.json').read_text())
cache_leads=read('data/processed/snapshots/source_extraction_20261019/council_education_source_leads_20261019.csv')
cache_pages=read('data/processed/snapshots/source_extraction_20261019/candidate_background_source_pages_20261019.csv')
assert len(cache_leads)==343 and not cache_leads.duplicated(['candidate_key','source_url','upstream_claim_raw']).any()
cache_index=cache_leads.set_index(['candidate_key','source_url','upstream_claim_raw']).to_dict('index')
for rec in cache_pages.to_dict('records'):
 assert rec['source_sha256']==lookup[rec['fetch_url']]['sha256']
 assert norm(rec['candidate_name']) in norm(rec['page_text'].replace('\\n','\n'))
parse_errors=cache_meta['parse_errors'];html={}
docs={r['url']:None for r in manifest if r['content_kind']=='pdf' and r['http_status']=='200' and r['url'] not in parse_errors}
assert len(docs)==cache_meta['saved_pdf_documents']

for r in manifest:
 if r['http_status']!='200' or not r['path']:continue
 try:
  if r['content_kind']=='pdf':
   continue  # Previously executed, hash-bound page extraction is reused; do not parse every PDF again.
  elif 'text/html' in r['content_type'].lower():
   s=BeautifulSoup((ROOT/r['path']).read_bytes().decode('utf-8'),'html.parser')
   for node in s(['script','style','noscript']):node.decompose()
   html[r['url']]=s
 except Exception as err:parse_errors[r['url']]=type(err).__name__+': '+str(err)
leads=[];page_rows=cache_pages.to_dict('records');profiles=[];evidence=[];social=[]
def add_evidence(p,field,value,rec,locator,tier,year='2026'):
 if not value:return
 value=str(value).replace('\r','').replace('\n','\\n')
 evidence.append(dict(candidate_key=p['candidate_key'],candidate_name=p['candidate_name'],county_city=p['county_city'],office_type=p['office_type'],field=field,value_raw=value,source_url=rec['url'],source_file=rec['path'],source_sha256=rec['sha256'],source_locator=locator,source_year=year,evidence_tier=tier,identity_status='source_name_and_candidate_context_reviewed_pending_independent_identity_check',checked_date='2026-10-19'))
manual={
 '陳彥廷':(['台大光電碩士'],['家樂福 15 年資深經歷']),
 '李明燦':(['清大電機碩士','成大電機學士','嘉義高中／水上國中／大同國小'],['台灣、美國與中國 3 地 25 年半導體電子業研發與品保主管','曾任聯電集團處長','唐恩書店創辦人','台灣模型車博物館 TMCM 創辦人']),
 '錢康明':(['國立台東大學兒童文學碩士','國立藝術學院（臺北藝術大學）戲劇系 藝術學學士'],['前新竹市文化局局長','台中國家歌劇院評鑑委員暨評議委員']),
 '林妤芬':(['美國加州大學柏克萊分校','法學碩士（LL.M.）','輔仁大學財經法律系','法學士'],['新竹市政府廉政會報委員','新竹縣警局諮詢委員','弘準國際法律事務所','主持律師']),
 '陳又新':(['台大法律系及社工系雙學位學士','交大科法所碩士'],[]),
 '林彥':(['國立臺灣科技大學 工學院學士','國立高雄應用科技大學 商學院學士','國立高雄科技大學 商學院碩士'],['曾於 高雄市政府 負責觀光旅宿管理','也曾在 高雄榮民總醫院 推動各行政區與醫療單位的公共設施改善','並於 國立傳統藝術中心（臺灣豫劇團）負責人事與總務等公共事務']),
 '陳力新':(['國立政治大學 EMBA 碩士','台大 E 勢泮後 EMBA 結業'],['國際企業負責人']),
 '林鈺梅':(['國立中興大學企業管理學系','國立中興大學應用經濟學系碩士在職專班','順利完成碩士學位'],['日月光集團（ASE Group）旗下環隆電氣（USI）任職','科技業服務七年']),
 '許纘融':(['中國醫藥大學製藥碩士畢業','成都中醫藥大學臨床醫學畢業'],['現於南投在地擔任專業看護與按摩師傅']),
 '蕭告':(['國立臺灣大學法律學系司法組畢業','加州大學舊金山法學院（UC Hastings）交換學生'],['蕭告律師事務所主持律師']),
}
seen_profiles=set()
for lead in plan['education_leads']:
 r=lookup[lead['fetch_url']];p=master[master.candidate_key.eq(lead['candidate_key'])];assert len(p)==1;p=p.iloc[0].to_dict()
 host=urllib.parse.urlparse(r['url']).hostname or '';primary_gov=host.endswith('.gov.tw');name=lead['candidate_name'];nname=norm(name)
 requested=urllib.parse.parse_qs(urllib.parse.urlparse(lead['upstream_source_url']).fragment).get('page',[''])[0]
 cached=cache_index[(p['candidate_key'],lead['upstream_source_url'],json.dumps(lead['upstream_claim'],ensure_ascii=False))]
 assert cached['source_sha256']==r['sha256'] and cached['fetch_url']==r['url']
 hits=[int(v.strip()) for v in cached['actual_name_pages'].split('|') if v.strip()]
 selected=[];pages=[]  # Cached page_rows already contain the exact saved extraction.
 page_name_status=cached['page_name_status']
 htext=html[r['url']].get_text(' ',strip=True) if r['url'] in html else ''
 html_name=bool(htext and nname in norm(htext))
 leads.append(dict(candidate_key=p['candidate_key'],candidate_name=name,county_city=p['county_city'],electoral_district=p['electoral_district'],upstream_claim_raw=json.dumps(lead['upstream_claim'],ensure_ascii=False),upstream_source_type=lead['upstream_source_type'],upstream_source_year=str(lead['upstream_source_year'] or ''),source_url=lead['upstream_source_url'],fetch_url=r['url'],http_status=r['http_status'],source_file=r['path'],source_sha256=r['sha256'],document_is_government=str(primary_gov),requested_page=requested,actual_name_pages=' | '.join(map(str,hits)),page_name_status=page_name_status,html_name_present=str(html_name),validation_status='source_saved_name_found_fields_and_identity_pending' if hits or html_name else 'source_access_or_name_check_pending',parse_error=parse_errors.get(r['url'],''),upstream_repository='weryk153/legislator-background',upstream_commit=raw_index_manifest[0]['commit']))
 for no in selected:
  page_rows.append(dict(candidate_key=p['candidate_key'],candidate_name=name,county_city=p['county_city'],source_url=lead['upstream_source_url'],fetch_url=r['url'],source_file=r['path'],source_sha256=r['sha256'],page=no,page_text=pages[no-1].replace('\r','').replace('\n','\\n'),page_role='whole_source_page_not_segmented_candidate_fields',identity_status='historical_name_match_pending_independent_linkage',government_source=str(primary_gov)))
 if lead['upstream_source_type']!='website' or not html_name or p['candidate_key'] in seen_profiles:continue
 s=html[r['url']];ed=[];ex=[];tier='candidate_website_self_declaration_not_degree_certification';year='2026';loc='manually selected literal source spans'
 if 'pridewatch.tw' in host:
  ednode=s.select_one('.views-field-field-education .field-content');exnode=s.select_one('.views-field-field-qualifications .field-content')
  ed=[ednode.get_text(' | ',strip=True)] if ednode else [];ex=[exnode.get_text(' | ',strip=True)] if exnode else []
  tier='civil_society_candidate_profile_declaration_not_independent_certification';loc='.views-field-field-education/.views-field-field-qualifications .field-content'
  current=s.select_one('.views-field-field-election-position-2026 .field-content');year='2026' if current and current.get_text(strip=True) else '2022_or_earlier_profile_not_currently_reconfirmed'
  for a in s.select('ul.candidate-social-links a[href]'):
   social.append(dict(candidate_key=p['candidate_key'],candidate_name=name,county_city=p['county_city'],account_url=a['href'],source_url=r['url'],source_file=r['path'],status='candidate_specific_social_block_link_target_content_pending'))
 elif host=='newpowerparty.tw':
  for heading in s.select('h5'):
   label=heading.get_text(strip=True)
   if label not in ['學歷','經歷']:continue
   container=heading.find_parent('div',class_='e-con');node=container.select_one('.elementor-widget-text-editor') if container else None
   if node:(ed if label=='學歷' else ex).append(node.get_text(' | ',strip=True))
  tier='official_party_candidate_profile_self_declaration';loc='h5 學歷／經歷所在e-con之text-editor'
 elif name=='應佳妤':
  for block in s.select('[data-cy="profile-info-content"]'):
   if '【學歷】' not in block.get_text():continue
   mode=''
   for node in block.find_all('p'):
    t=node.get_text(' ',strip=True)
    if t in ['【學歷】','【經歷】']:mode=t;continue
    if t.startswith('【'):mode=''
    if t and mode:(ed if mode=='【學歷】' else ex).append(t)
  loc='profile-info-content中【學歷】／【經歷】段落'
 elif name in manual:
  ed,ex=manual[name]
  for value in ed+ex:assert norm(value) in norm(htext),(name,value)
 # The profile URL is useful even when no defensible field extraction exists.
 add_evidence(p,'profile_url',r['url'],r,'candidate/person background page with matching name',tier,year)
 if ed:add_evidence(p,'education',' | '.join(ed),r,loc,tier,year)
 if ex:add_evidence(p,'experience',' | '.join(ex),r,loc,tier,year)
 profiles.append(dict(candidate_key=p['candidate_key'],candidate_name=name,county_city=p['county_city'],electoral_district=p['electoral_district'],profile_url=r['url'],education_raw=' | '.join(ed),experience_raw=' | '.join(ex),source_file=r['path'],source_sha256=r['sha256'],source_year=year,evidence_tier=tier,identity_status='name_and_candidate_context_lead_pending_independent_verification'))
 seen_profiles.add(p['candidate_key'])
 if name=='林鈺梅':
  assert norm('民國81年10月17日鈺梅出生於臺中縣') in norm(htext)
  add_evidence(p,'birth_date','1992-10-17',r,'出生・成長：民國81年10月17日鈺梅出生於臺中縣','candidate_website_explicit_birth_date_self_declaration','2026')
leads=pd.DataFrame(leads);pages=pd.DataFrame(page_rows).drop_duplicates(['candidate_key','fetch_url','page']);profiles=pd.DataFrame(profiles);evidence=pd.DataFrame(evidence)
assert len(leads)==343 and leads.candidate_key.nunique()==341
assert profiles.candidate_key.is_unique
assert not evidence.duplicated(['candidate_key','field','source_url']).any()
def update(key,field,value,url,status,note):
 mask=fields.candidate_key.eq(key)&fields.field.eq(field);assert mask.sum()==1
 fields.loc[mask,['existing_value','evidence','status','note','next_source','updated_date']]=[value,url,status,note,'官方公報／獨立身分及欄位複核','2026-10-19']
 mask=master.candidate_key.eq(key);assert mask.sum()==1;master.loc[mask,field+'_status']=status
for r in evidence.to_dict('records'):
 if r['field'] in ['profile_url','education','birth_date']:
  field='profile' if r['field']=='profile_url' else r['field']
  update(r['candidate_key'],field,r['value_raw'],r['source_url'],'背景原文已保存，來源自述／人物對應待獨立複核',r['source_locator']+'；來源年：'+r['source_year']+'；非戶籍或學位認證。')
# Source discovery is real work, but do not turn a third-party claim into a verified education value.
extracted_education_keys=set(evidence[evidence.field.eq('education')].candidate_key)
for key,d in leads.groupby('candidate_key',sort=False):
 if key in extracted_education_keys:continue
 mask=fields.candidate_key.eq(key)&fields.field.eq('education');assert mask.sum()==1
 if fields.loc[mask,'existing_value'].iloc[0]!='':continue
 found=d.validation_status.eq('source_saved_name_found_fields_and_identity_pending').any()
 status='已找到學歷出處及姓名線索；欄位與身分待核對' if found else '已追查出處；存取或姓名確認仍待完成'
 sources=' | '.join(dict.fromkeys(d.source_url))
 update(key,'education','',sources,status,'保留第三方線索於council_education_source_leads，不把摘要直接填成已核實學歷。')

# Keep history-supported fields unchanged and separate new self-declared observations.
council['additional_profile_url']='';council['additional_profile_status']='';council['birth_date_self_declared']='';council['birth_date_self_declared_source']=''
for r in profiles.to_dict('records'):
 mask=council.candidate_key.eq(r['candidate_key']);assert mask.sum()==1
 council.loc[mask,['additional_profile_url','additional_profile_status']]=[r['profile_url'],r['identity_status']]
 if r['education_raw']:council.loc[mask,'education_raw']=r['education_raw']
 if r['experience_raw']:council.loc[mask,'experience_raw']=r['experience_raw']
 council.loc[mask,'background_source_url']=r['profile_url']
for r in evidence[evidence.field.eq('birth_date')].to_dict('records'):
 mask=council.candidate_key.eq(r['candidate_key']);council.loc[mask,['birth_date_self_declared','birth_date_self_declared_source']]=[r['value_raw'],r['source_url']]
# Preserve all mayor links as third-party leads, without upgrading to official verification.
channel=json.loads((ROOT/'data/raw/public_source_index_20261019/candidate-channels.json').read_text());mayor_leads=[]
for p in channel['candidates']:
 county=p['role'].replace('長','').replace('市','市')
 matches=master[master.candidate_name.map(norm).eq(norm(p['name'])) & master.office_type.str.contains('長')]
 if len(matches)!=1:continue
 person=matches.iloc[0]
 for link in p['links']:
  mayor_leads.append(dict(candidate_key=person.candidate_key,candidate_name=person.candidate_name,county_city=person.county_city,kind=link['kind'],url=link['url'],upstream_evidence_url=link.get('evidence',{}).get('url',''),upstream_note=link.get('evidence',{}).get('note',''),verification_status='third_party_discovery_lead_not_independently_reverified',upstream_repository='weryk153/legislator-background',upstream_commit=raw_index_manifest[0]['commit']))
mayor_leads=pd.DataFrame(mayor_leads)
social=pd.DataFrame(social,columns=['candidate_key','candidate_name','county_city','account_url','source_url','source_file','status']).drop_duplicates()
outputs={'data/processed/snapshots/source_extraction_20261019/council_education_source_leads_20261019.csv':leads,'data/processed/snapshots/source_extraction_20261019/candidate_background_source_pages_20261019.csv':pages,'data/processed/candidate_web_profiles_expansion_20261019.csv':profiles,'data/processed/candidate_background_evidence_expansion_20261019.csv':evidence,'data/processed/mayor_channel_discovery_leads_20261019.csv':mayor_leads,'data/processed/council_social_links_expansion_20261019.csv':social,'data/processed/snapshots/pre_government_round4_20261019/candidate_master_current.csv':master,'data/processed/snapshots/pre_government_round4_20261019/candidate_fields_current.csv':fields,'data/processed/snapshots/pre_government_round4_20261019/council_background_20261019.csv':council}
verification={}
for path,frame in outputs.items():
 assert path not in PINS and (ROOT/path).is_relative_to(ROOT/'data/processed')
 frame.to_csv(ROOT/path,index=False,encoding='utf-8-sig');d=read(path)
 assert len(d)==len(frame) and list(d.columns)==list(frame.columns)
 assert len((ROOT/path).read_text(encoding='utf-8-sig').splitlines())==len(d)+1,path
 verification[path]={'rows':len(d),'sha256':sha(ROOT/path)}
assert len(master)==1583 and master.candidate_key.is_unique and len(fields)==31660 and not fields.duplicated(['candidate_key','field']).any()
assert len(council)==1502 and council.birth_date_supported_history.ne('').sum()==690 and council.sex_supported_history.ne('').sum()==690
before=read(snap+'candidate_fields_current.csv').set_index(['candidate_key','field']);after=fields.set_index(['candidate_key','field'])
for field in ['sex','incumbency','first_time_this_office','first_time_any_office','administrative_experience','posts_Facebook','posts_Instagram','posts_Threads','followers']:
 pd.testing.assert_frame_equal(before.xs(field,level='field'),after.xs(field,level='field'))
summary={'retrieved_urls':len(manifest),'http_200':sum(r['http_status']=='200' for r in manifest),'saved_pdf_documents':len(docs),'education_lead_people':341,'education_leads':343,'source_name_found_people':int(leads[leads.validation_status.eq('source_saved_name_found_fields_and_identity_pending')].candidate_key.nunique()),'page_records':len(pages),'new_background_profile_people':len(profiles),'new_education_people':int(profiles.education_raw.ne('').sum()),'new_experience_people':int(profiles.experience_raw.ne('').sum()),'no_profile_after_expansion':int((council.moi_profile_url.eq('') & council.campaign_profile_url.eq('') & council.additional_profile_url.eq('')).sum()),'mayors_with_discovery_links':int(mayor_leads.candidate_key.nunique()),'mayor_link_records':len(mayor_leads),'source_specific_social_links':len(social),'parse_errors':parse_errors,'outputs':verification}
(ROOT/'data/code/source_expansion_20261019_verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "retrieved_urls": 290,
  "http_200": 265,
  "saved_pdf_documents": 171,
  "education_lead_people": 341,
  "education_leads": 343,
  "source_name_found_people": 252,
  "page_records": 171,
  "new_background_profile_people": 22,
  "new_education_people": 21,
  "new_experience_people": 20,
  "no_profile_after_expansion": 763,
  "mayors_with_discovery_links": 62,
  "mayor_link_records": 85,
  "source_specific_social_links": 9,
  "parse_errors": {
    "https://eebulletin.cec.gov.tw/111/14%E5%B1%8F%E6%9D%B1%E7%B8%A3/02%E7%B8%A3%E8%AD%B0%E5%93%A1/%E7%AC%AC1%E9%81%B8%E5%8D%801.pdf": "PdfiumError: Failed to load document (PDFium: Data format error).",
    "https://eebulletin.cec.gov.tw/111/03%E6%96%B0%E5%8C%97%E5%B8%82/02%E5%B8%82%E8%AD%B0%E5%93%A1/%E5%8D%80%E5%9F%9F%E8%AD%B0%E5%93%A1%E9%81%B8%E8%88%89%E5%85%AC%E5%A0%B1%E7%AC%AC6%E9%81%B8%E8%88%89%E5%8D%80.pdf": "PdfiumError: Failed to load document (PDFium: Data format error).",
    "https://eebulletin.cec.gov.tw/111/11%E5%8D%97%E6%8A%95%E

## 社群入口接續核對
七個具名候選人hub／競選支持頁已查看，與保存的競選官網及公民團體人物專屬社群區塊接回帳號工作單。排除貼文、分享與群組，不推定目標內容可完整讀取。

In [2]:

# Extend the same pipeline with candidate-owned hub and campaign-site account links.
from urllib.parse import urlparse,urlunparse,parse_qs,urlencode
SOCIAL_PINS={'data/raw/candidate_hubs_20261019/manifest.json': '23873e6e33282927cbb2c55f9b33a0faa901d2bb61cf4ec38f563098bcceb0b1', 'data/processed/candidate_social_account_evidence_20261019.csv': 'b15e1bf88876de049533f381771920c3c560e18eef5c53e04fb8142cbedc5ff9'}
for p,h in SOCIAL_PINS.items():assert sha(ROOT/p)==h,p
hubs=json.loads((ROOT/'data/raw/candidate_hubs_20261019/manifest.json').read_text())
for r in hubs:
 if r['path']:assert sha(ROOT/r['path'])==r['sha256']
old_social=read('data/processed/candidate_social_account_evidence_20261019.csv')
social_rows=[];unresolved=[]
for rec in old_social.to_dict('records'):
 if re.search(r'/(?:share|sharer|posts|post|reel|reels|groups|videos|watch)(?:/|\?|$)',rec['account_url']):
  unresolved.append(dict(candidate_key=rec['candidate_key'],candidate_name=rec['candidate_name'],url=rec['account_url'],source_url=rec['source_url'],reason='legacy_share_or_content_link_excluded_from_account_list'))
 else:social_rows.append(rec)
def plat(url):return {'facebook.com':'Facebook','fb.com':'Facebook','instagram.com':'Instagram','threads.net':'Threads','threads.com':'Threads'}.get(urlparse(url).netloc.lower().removeprefix('www.'),'')
def canonical(url):
 u=urlparse(url);host=u.netloc.lower().removeprefix('www.')
 host={'fb.com':'facebook.com','threads.net':'threads.com'}.get(host,host)
 query=urlencode({'id':parse_qs(u.query)['id'][0]}) if host=='facebook.com' and u.path=='/profile.php' and 'id' in parse_qs(u.query) else ''
 return urlunparse(('https','www.'+host,u.path.rstrip('/'),'',query,''))
def add_social(person,url,rec,kind,locator):
 platform=plat(url)
 if not platform:return
 path=urlparse(url).path
 if re.search(r'/(?:share|sharer|sharer.php|share.php|post|posts|reel|reels|groups|videos|watch|p/[^/]+/[^/]+)',path):
  unresolved.append(dict(candidate_key=person['candidate_key'],candidate_name=person['candidate_name'],url=url,source_url=rec['url'],reason='share_or_content_or_group_link_not_resolved_to_personal_account'));return
 if not path or path in ['/p','/login','/intent']:return
 social_rows.append(dict(candidate_key=person['candidate_key'],candidate_name=person['candidate_name'],county_city=person['county_city'],office_type=person['office_type'],platform=platform,account_url=canonical(url),source_url=rec['url'],source_file=rec['path'],source_locator=locator,evidence_status=kind,target_content_status='not_verified_not_posts_captured',checked_date='2026-10-19'))
# Only the candidate-specific social block on civil-society profile pages.
for r in read('data/processed/council_social_links_expansion_20261019.csv').to_dict('records'):
 person=master[master.candidate_key.eq(r['candidate_key'])].iloc[0].to_dict()
 add_social(person,r['account_url'],{'url':r['source_url'],'path':r['source_file']},'candidate_specific_civil_society_profile_link','ul.candidate-social-links')
# Campaign websites: source name has been checked; filter content/share/group links.
for item in channel['candidates']:
 persons=master[master.candidate_name.map(norm).eq(norm(item['name']))&master.office_type.str.contains('長')]
 if len(persons)!=1:continue
 person=persons.iloc[0].to_dict()
 for link in item['links']:
  if link['kind'] not in ['campaign','website']:continue
  s=html.get(link['url']);rec=lookup.get(link['url'])
  if s is None or rec is None:continue
  assert norm(item['name']) in norm(s.get_text(' ',strip=True))
  for a in s.select('a[href]'):add_social(person,a['href'],rec,'candidate_campaign_website_link','campaign site navigation/footer social profile links; content URLs excluded')
# Named hub/support pages: all seven were inspected for name and election context.
for r in hubs:
 if r['status']!='200':continue
 persons=master[master.candidate_name.eq(r['candidate_name'])&master.office_type.str.contains('長')];assert len(persons)==1
 person=persons.iloc[0].to_dict();s=BeautifulSoup((ROOT/r['path']).read_bytes().decode('utf-8'),'html.parser')
 assert norm(person['candidate_name']) in norm(s.get_text(' ',strip=True))
 for a in s.select('a[href]'):add_social(person,a['href'],r,'candidate_hub_or_support_page_self_link','named candidate hub or campaign support page social links')
social_current=pd.DataFrame(social_rows).drop_duplicates(['candidate_key','platform','account_url','source_url']).sort_values(['county_city','candidate_name','platform','account_url'])
assert social_current.candidate_key.isin(master.candidate_key).all()
assert not social_current.account_url.str.contains('/share/|/groups/|/posts/|/post/|/reel/|moi.gov.tw').any()
work=[]
for p in master.to_dict('records'):
 for platform in ['Facebook','Instagram','Threads']:
  d=social_current[social_current.candidate_key.eq(p['candidate_key'])&social_current.platform.eq(platform)];urls=' | '.join(dict.fromkeys(d.account_url));sources=' | '.join(dict.fromkeys(d.source_url))
  work.append(dict(candidate_key=p['candidate_key'],candidate_name=p['candidate_name'],county_city=p['county_city'],office_type=p['office_type'],platform=platform,official_urls=urls,discovered_on=sources,account_status='candidate_specific_source_link_target_content_pending' if urls else 'not_yet_found_not_absent',posts_status='not_collected_not_zero',window_count='',followers_start='',followers_end='',updated_date='2026-10-19'))
  if urls:
   update(p['candidate_key'],platform,urls,sources,'人物頁／競選頁已列帳號連結；目標內容待核對','保留來源類型；公民團體人物頁非政府認證。貼文窗口及蒐集暫緩。')
   mask=fields.candidate_key.eq(p['candidate_key'])&fields.field.eq(platform);fields.loc[mask,'next_source']='帳號首頁及候選人原頁交叉核對；未蒐集貼文不填零'
work=pd.DataFrame(work);assert len(work)==4749 and not work.duplicated(['candidate_key','platform']).any()
unresolved=pd.DataFrame(unresolved).drop_duplicates(['candidate_key','url','source_url'])
extra={'data/processed/snapshots/pre_bulletin_round5_20261019/candidate_social_account_evidence_current.csv':social_current,'data/processed/snapshots/pre_bulletin_round5_20261019/meta_candidate_account_worklist_current.csv':work,'data/processed/social_unresolved_links_20261019.csv':unresolved,'data/processed/snapshots/pre_government_round4_20261019/candidate_master_current.csv':master,'data/processed/snapshots/pre_government_round4_20261019/candidate_fields_current.csv':fields}
for path,frame in extra.items():
 frame.to_csv(ROOT/path,index=False,encoding='utf-8-sig');saved=read(path);assert len(saved)==len(frame) and list(saved.columns)==list(frame.columns)
 assert len((ROOT/path).read_text(encoding='utf-8-sig').splitlines())==len(saved)+1
 verification[path]={'rows':len(saved),'sha256':sha(ROOT/path)}
summary['social_candidates_with_source_links']=int(social_current.candidate_key.nunique())
summary['social_candidate_platform_pairs']=int(social_current[['candidate_key','platform']].drop_duplicates().shape[0])
summary['social_by_platform']=social_current[['candidate_key','platform']].drop_duplicates().platform.value_counts().to_dict()
summary['social_hub_pages_read']=sum(r['status']=='200' for r in hubs)
summary['outputs']=verification
(ROOT/'data/code/source_expansion_20261019_verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
def profile_links(d):return '、'.join(f'[{i+1}]({url})' for i,url in enumerate(dict.fromkeys(d.account_url))) or '待補查'
lines=['# 候選人社群入口（2026-10-19）','','本清單是可點選的帳號入口。候選人／政黨／競選頁或人物頁列出的連結已保存來源；尚未全面核對目標帳號內容，未蒐集完整貼文。未找到不代表沒有帳號。第三方單純提供但未回原頁核對的62位縣市長連結線索另見[新來源工作單](source_expansion_20261019.md)。','',f"目前涵蓋 **{summary['social_candidates_with_source_links']} 位候選人、{summary['social_candidate_platform_pairs']} 組候選人×平台**。來源範圍不均，不能用這份已找到名單比較黨派社群使用率。",'','## 蔣萬安、沈伯洋與薛兆基','']
for name in ['蔣萬安','沈伯洋','薛兆基']:
 p=master[master.candidate_name.eq(name)].iloc[0];d=social_current[social_current.candidate_key.eq(p.candidate_key)]
 lines += [f'### {name}（{p.county_city}／{p.office_type}）','','| Facebook | Instagram | Threads |','| --- | --- | --- |','| '+' | '.join(profile_links(d[d.platform.eq(platform)]) for platform in ['Facebook','Instagram','Threads'])+' |','','來源：'+'、'.join(f'[{i+1}]({url})' for i,url in enumerate(dict.fromkeys(d.source_url))),'']
lines+=['## 其他已找到入口的候選人','','| 候選人 | 縣市／公職 | Facebook | Instagram | Threads | 核對來源 |','| --- | --- | --- | --- | --- | --- |']
for key,d in social_current.groupby('candidate_key',sort=False):
 p=master[master.candidate_key.eq(key)].iloc[0]
 if p.candidate_name in ['蔣萬安','沈伯洋','薛兆基']:continue
 sources='、'.join(f'[{i+1}]({url})' for i,url in enumerate(dict.fromkeys(d.source_url)))
 lines.append('| '+' | '.join([p.candidate_name,p.county_city+'／'+p.office_type]+[profile_links(d[d.platform.eq(platform)]) for platform in ['Facebook','Instagram','Threads']]+[sources])+' |')
lines+=['','## 核對界線','','- 民進黨候選人物件、候選人網站、Portaly／Linktree本人連結頁、具名競選支持頁與公民團體人物頁分別保留來源類型。這些不是Meta授權或政府身分認證。','- 薛兆基目前有Facebook來源連結；Instagram與Threads仍待找到可核對入口，未填成沒有帳號。','- 分享網址、貼文、Reels與Facebook群組不當作個人帳號；尚未解析者另存social_unresolved_links_20261019.csv。','- 多個Facebook名稱／數字網址先保留，未確認是否同頁前不任意合併。','- 帳號連結可點選不代表本研究已保存完整貼文，貼文與追蹤數未知不填零。','- 直接資料入口：[目前社群工作單](../data/processed/snapshots/pre_bulletin_round5_20261019/meta_candidate_account_worklist_current.csv)、[逐來源帳號證據](../data/processed/snapshots/pre_bulletin_round5_20261019/candidate_social_account_evidence_current.csv)。']
(ROOT/'report/candidate_social_links_pre_bulletin_round5_20261019.md').write_text('\n'.join(lines)+'\n')
print(json.dumps({key:summary[key] for key in ['social_candidates_with_source_links','social_candidate_platform_pairs','social_by_platform','social_hub_pages_read']},ensure_ascii=False,indent=2))


{
  "social_candidates_with_source_links": 63,
  "social_candidate_platform_pairs": 158,
  "social_by_platform": {
    "Facebook": 59,
    "Instagram": 56,
    "Threads": 43
  },
  "social_hub_pages_read": 7
}


2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。